## Programming part of Homework 4 (Data Structures, Fall 2025)

## Name: 邱冠勛
## Student ID Number: 113590021

### Programming problem 1
**Univariate polynomial** of degree $d$ has the form $$c_dx^d+c_{d-1}x^{d-1}+\cdots + c_2x^2+c_1x+c_0,$$ where $c_d\not= 0$. The $c_i$'s are the \emph{coefficients}, and $d, d-1, \cdots$ are the \emph{exponents}. By definition, $d$ is a nonnegative integer. In this exercise, we assume that all $c_i$s are integers. We represent each polynomial as a *linear list* of coefficients and would like to have some operations (functions) on the polynomials. The first node in the list represents the first terms in the polynomial, the second node represents the second terms, and so forth.

Each node contains three fields: *the term's coefficient*, *the term's power*, and *a pointer to the next term*. Write a Python program, that first reads the input file, `inFile.txt`, which has three lines and then performs the indicated operation. The first line is an integer representing the operation defined as below. The second line is the first polynomial and the next line is the second polynomial. The input polynomial, say $4x^3-2x+1$, will be represented as `4x^3-2x+1`. The functions include the following operations:
1. `add`: Add two input polynomials.
2. `subtract`: Subtract the second polynomial from the first one.
3. `multiply`: Multiply two polynomials.
4. `divide`: Divide the first polynomial by the second one and return the quotient.
The input file thus may be

```
2
4x^3-2x+1
3x^2+x+4
```

Your output will be `4x^3-3x^2-3x-3`. Please see the running example in the end of this template.

Python has a built-in package called **re**, which can be used to work with Regular Expressions and provides regular expression matching operations similar to those found in Perl. One can use this package for parsing the input strings. For more details, please refer to
[PYTHON Regular Expression](https://www.w3schools.com/python/python_regex.asp) and 
[Python RegEx](https://docs.python.org/3/library/re.html).

First, we build up the linked list structure for representing polynomials. Two classes will be defined: `Node` and `Poly_List`.

In [5]:
# Build up the linked list structure for representing polynomials
import re
from typing import Optional # When you have imported the "re" module, you can start using regular expressions

# Define the class of node in the linked list used for polynomial representation
# Node class 
class Node:
    def __init__(self, c: 'float', exp: 'int'):
        self.coefficient: float = float(c)
        self.exponential: int = int(exp)
        self.next: Node | None = None
        
    # get the coefficient in this node
    def getCoefficient(self):
        return self.coefficient
    
    # get the exponent in this node
    def getExponential(self):
        return self.exponential
    
    # get the next node
    def getNext(self):
        return self.next
    
    # set the coefficient and exponent to this node
    def setData(self,c,exp):
        self.coefficient = c
        self.exponential = exp

    # set the coefficient to this node only
    def setCoefficient(self,c):
        self.coefficient = c

    # set the exponent to this node only
    def setExponential(self,exp):
        self.exponential = exp

    # assign the next node to this node 
    def setNext(self,newnext):
        self.next= newnext

    def __repr__(self) -> str:
        return f"{self.coefficient}^{self.exponential}"

# Define the class of the linked list used for polynomial representation
# List class 
class Poly_List:
    def __init__(self):
        self.head: Node | None = None
        self.tail: Node | None = None

    # methods for managing the list
    def isEmpty(self):
        if self.head is None and self.tail is None:
            return True
        
        return False

    def size(self):
        length = 0
        ptr = self.head

        while ptr is not None:
            ptr = ptr.next
            length += 1

        return length

    def isHead(self, node):
        return node is self.head 

    def isTail(self, node):
        return node is self.tail 
    
    # get the head of the list
    def getHead(self):
        return self.head

    # get the tail of the list
    def getTail(self):
        return self.tail
    
    # set the head of the list
    def setHead(self, node):
        self.head = node

    # set the tail of the list
    def setTail(self, node):
        self.tail = node

    # get the degree of the polynomial
    def polyDegree(self):
        return self.size()

    # insert a term (node) after node p
    def insertAfter(self, p: 'Node', c: 'float', exp: 'int'):
        newNode = Node(c, exp)
        
        newNode.next = p.next
        p.next = newNode

    # insert a term (node) at head
    def insertAtHead(self, c: 'float', exp: 'int'):
        newNode = Node(c, exp)

        newNode.next = self.head
        self.head = newNode

    # insert a term (node) at tail
    def insertAtTail(self, c: 'float', exp: 'int'):
        newNode = Node(c, exp)

        if self.tail is None:
            self.head = newNode
            self.tail = newNode
            return
        
        self.tail.next = newNode
        self.tail = newNode

    # delete a term (node) at head
    def deleteAtHead(self):
        assert (self.head is not None)

        if self.head is self.tail:
            self.head = None
            self.tail = None
            return

        self.head = self.head.next
        
    # Method for adding the missing terms and may be used for division
    def paddingPoly(self):
        if self.head is None:
            return
        
        highest_exp = self.head.exponential
        current_node = self.head

        highest_exp -= 1 # Head for next node
        while (highest_exp >= 0):
            if current_node.next is None or \
                current_node.next.exponential != highest_exp:
                
                self.insertAfter(current_node, 0, highest_exp)           

            highest_exp -= 1

            if current_node.next is None:
                raise AssertionError("Next node is empty during padding (unexpected)")

            current_node = current_node.next

        # Update tail node
        t = self.head
        while t.next is not None:
            t = t.next
        self.tail = t

    # This method is used for multiplying the polynomial by a constant m or
    # lifting all terms by a degree d
    def timeConst_liftDegree(self, m: float, d: int):
        current_node = self.head
        
        while (current_node != None):
            current_node.coefficient *= m
            current_node.exponential += d
            current_node = current_node.next
        
        
    # Method to verify the degree of polynomial for getting rid of the higher
    # terms with 0 as coefficients

    # This method returns a copy of the polynomail with a new list
    def copy(self) -> 'Poly_List':
        new_poly = Poly_List()

        current_node = self.head

        while current_node is not None:
            new_poly.insertAtTail(current_node.coefficient, current_node.exponential)
            current_node = current_node.next

        return new_poly
    
    # This is used to print the list for represented polynomial
    def printPoly_List(self):
        current_node = self.head

        print("[ ", end="")
        while current_node is not None:
            print(current_node, end=" ")
            current_node = current_node.next

        print(" ]")

    # This prints the polynomial in a given format
    def printPolynomial(self):
        current_node = self.head

        while current_node is not None:
            if current_node is self.head:
                sign = "" if current_node.coefficient > 0 else "-"
            else:
                sign = "+" if current_node.coefficient > 0 else ""

            if current_node.exponential == 1:
                print(f"{sign}{current_node.coefficient:.1f}x", end="")
            elif current_node.exponential == 0:
                print(f"{sign}{current_node.coefficient:.1f}", end="")
            else:
                print(f"{sign}{current_node.coefficient:.1f}x^{current_node.exponential}", end="")
                
            current_node = current_node.next
        
        print()

Then, we may provide the functions for helping read and parse the input file to have the input operation and polynnomials. 
The `read_lines()` function reads the lines into and returns a list of strings. 
Function `read_string(s)` parses an input string to a polynomial with ***linked list representation***. 

In [6]:
# functions for reading and parsing the input file to have the input polynnomials and operation 
# function for reading lines in the input text file into a list of strings      
def read_lines():
    with open("inFile.txt", "r") as f:
        return [i.replace("\n", "") for i in f.readlines()]
    
# function for parsing the line into polynomial with linked list representation 
def read_string(s: str):
    exp = r"(?=.)(\+|\-)?(\d+)?(x)?(\^\d+)?"
    result = re.findall(exp, s)

    poly_list = Poly_List()

    for i in result:
        term = ''.join(i)
        numbers = term.split("^")

        if "^" not in term:
            exponent = 1 if "x" in numbers[0] else 0
        else:
            exponent = int(numbers[1])
        
        if numbers[0] == "+x" or numbers[0] == "-x" or numbers[0] == "x":
            constant = float(numbers[0].replace("x", "1"))
        else:
            constant = float(numbers[0].replace("x", ""))

        poly_list.insertAtTail(constant, exponent)
    return poly_list

Below, the functions for polynomial operations with two input polynomials are provided:
1. `add()`: Add two input polynomials.
2. `subtract()`: Subtract the second polynomial from the first one.
3. `multiply()`: Multiply two polynomials.
4. `divide()`: Divide the first polynomial by the second one and return the quotient and remainder.

**Note that** since `divide()` returns two resulting polynomails. We therefore have all the functions for operations return two polynomials. If there is only one resulting polynomial, we use `None` object for the second polynomail to return.

In [7]:
# functions for polynomial operations
# adding two polynomials
def add(poly1: 'Poly_List', poly2: 'Poly_List'):
    result = Poly_List()
    p1 = poly1.head
    p2 = poly2.head

    while p1 is not None and p2 is not None:
        if p1.exponential > p2.exponential:
            result.insertAtTail(p1.coefficient, p1.exponential)
            p1 = p1.next
        elif p1.exponential < p2.exponential:
            result.insertAtTail(p2.coefficient, p2.exponential)
            p2 = p2.next
        else:
            coeff_sum = p1.coefficient + p2.coefficient
            if coeff_sum != 0:
                result.insertAtTail(coeff_sum, p1.exponential)
            p1 = p1.next
            p2 = p2.next

    while p1 is not None:
        result.insertAtTail(p1.coefficient, p1.exponential)
        p1 = p1.next

    while p2 is not None:
        result.insertAtTail(p2.coefficient, p2.exponential)
        p2 = p2.next

    return result, None

# substracting poly2 from poly1 
def substract(poly1, poly2):
    result = Poly_List()
    p1 = poly1.head
    p2 = poly2.head

    while p1 is not None and p2 is not None:
        if p1.exponential > p2.exponential:
            result.insertAtTail(p1.coefficient, p1.exponential)
            p1 = p1.next
        elif p1.exponential < p2.exponential:
            result.insertAtTail(-p2.coefficient, p2.exponential)
            p2 = p2.next
        else:
            coeff_diff = p1.coefficient - p2.coefficient
            if coeff_diff != 0:
                result.insertAtTail(coeff_diff, p1.exponential)
            p1 = p1.next
            p2 = p2.next

    while p1 is not None:
        result.insertAtTail(p1.coefficient, p1.exponential)
        p1 = p1.next

    while p2 is not None:
        result.insertAtTail(-p2.coefficient, p2.exponential)
        p2 = p2.next

    return result, None

# multiplying two polynomials
def multiply(poly1, poly2):
    result = Poly_List()
    p1 = poly1.head

    while p1 is not None:
        temp = Poly_List()
        p2 = poly2.head
        while p2 is not None:
            coeff = p1.coefficient * p2.coefficient
            exp = p1.exponential + p2.exponential
            temp.insertAtTail(coeff, exp)
            p2 = p2.next
        result, _ = add(result, temp)
        p1 = p1.next

    return result, None
    
def divide(poly1: 'Poly_List', poly2: 'Poly_List'):
    # Make copies to avoid modifying originals
    dividend = poly1.copy()
    divisor = poly2.copy()
    quotient = Poly_List()

    # Ensure both polynomials are padded (all exponents present)
    dividend.paddingPoly()
    divisor.paddingPoly()

    while dividend.head is not None and dividend.head.exponential >= divisor.head.exponential:
        # Leading term division
        coeff = dividend.head.coefficient / divisor.head.coefficient
        exp = dividend.head.exponential - divisor.head.exponential
        term = Poly_List()
        term.insertAtTail(coeff, exp)
        quotient.insertAtTail(coeff, exp)

        # Multiply divisor by this term and subtract from dividend
        temp = divisor.copy()
        temp.timeConst_liftDegree(coeff, exp)
        dividend, _ = substract(dividend, temp)

        # Remove leading zero terms if any
        while dividend.head is not None and abs(dividend.head.coefficient) < 1e-8:
            dividend.deleteAtHead()

    remainder = dividend
    return quotient, remainder


Last, we perform the operations according to the input file, `inFile.txt`. Function `poly_operation()` can be as the main program entry and first derive the operation with the derived list of strings from function `read_lines()`. Then, `operation_selection()` is called to perform the corresponding operation. **Note that** there will be two polynomials returned for each operation. Last, it prints out the result. The whole program will be executed by calling `poly_operation()` with the input file, `inFile.txt`. One can change the content in the input file for different cases. 

***Basically, the following cell can be kept without change if you would like to follow it for programming. Of course, you can have your own code for this part. However, the function name of the main program entry, `poly_operation()` can not be changed.***

In [8]:
# main program area
# function to call the corresponding operation and return two polynomial
def operation_selection(operation, poly1, poly2):
    switcher = {
        1: add,
        2: substract,
        3: multiply,
        4: divide,
    }
    # Get the function from switcher dictionary

    func = switcher.get(operation, lambda: "nothing")
    

    # Execute the function
    return func(poly1,poly2)

# program entry
# function for starting the task
def poly_operation():
    #
    # read the input information from the default input text file
    #
    strings=read_lines()

    #
    # obtain the operation: 1. add; 2. substract; 3. Multiply; 4. Divide
    # and print it out
    #
    operation=int(strings[0])
    operations={
        1: 'add',
        2: 'substract',
        3: 'multiply',
        4: 'divide'
    }
    print(strings[1], operations.get(operation), strings[2])

    #
    # parse strings 1 and 2 to derive the input polynomials and represent them with
    # linked lists
    #
    poly1=read_string(strings[1])
    poly2=read_string(strings[2])

    #
    # perform the operation and two polynomials are returned.
    #
    r1, r2=operation_selection(operation, poly1, poly2)

    #
    # print out the result
    #
    if (operation==4):
        print("The quotient is:", end="")
        r1.printPolynomial()
        print("The remainder is:", end="")
        r2.printPolynomial()
    else:
        print("The result is:", end="")
        r1.printPolynomial()        

# execute the program with the input file inFile.txt
poly_operation()

4x^3-2x+1 substract -3x^2+x+4
The result is:4.0x^3+3.0x^2-3.0x-3.0
